# 1. CNN для классификации изображений



## 1. Настройка


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
FAST_MODE = True  # False: полные train/test и больше эпох
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device, "| FAST_MODE:", FAST_MODE)


## 2. Dataset: что именно мы загружаем — 8 минут

`Dataset` — индексируемая коллекция: `len(dataset)` сообщает размер, а `dataset[i]` возвращает
`(image, class_index)`. `torchvision.datasets.EuroSAT` хранит RGB-версию EuroSAT. В ней нет официального
train/validation split, поэтому создадим **стратифицированное** разбиение: каждый класс представлен в обеих частях.

Спутниковые снимки заметно отличаются от обычных фотографий: ориентация часто не несёт смысла, а текстуры и
пространственные паттерны важнее привычных объектов.


In [ ]:
DATA_DIR = Path("data")
base_ds = datasets.EuroSAT(DATA_DIR, download=True)
class_names = base_ds.classes
targets_np = np.asarray(base_ds.targets)
print("Размер:", len(base_ds), "| классов:", len(class_names))
for class_id, name in enumerate(class_names):
    print(f"{class_id:2d} {name:>22}: {(targets_np == class_id).sum():4d}")


In [ ]:
# Визуализация исходных PIL-изображений до tensor/normalize.
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for class_id, ax in enumerate(axes.flat):
    index = int(np.flatnonzero(targets_np == class_id)[0])
    image, label = base_ds[index]
    ax.imshow(image); ax.set_title(class_names[label]); ax.axis("off")
plt.suptitle("По одному исходному примеру каждого класса")
plt.tight_layout(); plt.show()


## 3. Transforms и аугментации

Transform вызывается внутри `Dataset.__getitem__`. Train-transform случайный и создаёт новый вариант при каждом
обращении; validation-transform детерминирован. Для спутниковых кадров допустимы горизонтальные/вертикальные
отражения и повороты на 90° — в отличие, например, от цифр 6 и 9.

`ToTensor()` переводит PIL HWC `uint8` в CHW `float32` диапазона `[0,1]`; `Normalize` стандартизует каналы.


In [ ]:
EURO_MEAN = (0.344, 0.380, 0.408)
EURO_STD = (0.203, 0.136, 0.115)
train_transform = transforms.Compose([
    transforms.RandomVerticalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomChoice([transforms.RandomRotation((0, 0)),
                             transforms.RandomRotation((90, 90)),
                             transforms.RandomRotation((180, 180)),
                             transforms.RandomRotation((270, 270))]),
    transforms.ToTensor(),
    transforms.Normalize(EURO_MEAN, EURO_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(EURO_MEAN, EURO_STD),
])

train_source = datasets.EuroSAT(DATA_DIR, transform=train_transform)
val_source = datasets.EuroSAT(DATA_DIR, transform=eval_transform)
rng_split = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for class_id in range(len(class_names)):
    indices = np.flatnonzero(targets_np == class_id)
    rng_split.shuffle(indices)
    if FAST_MODE:
        train_indices.extend(indices[:600]); val_indices.extend(indices[600:750])
    else:
        cut = int(0.8 * len(indices)); train_indices.extend(indices[:cut]); val_indices.extend(indices[cut:])
train_ds, val_ds = Subset(train_source, train_indices), Subset(val_source, val_indices)
print("train:", len(train_ds), "| val:", len(val_ds), "| overlap:", len(set(train_indices) & set(val_indices)))


In [ ]:
def denormalize(batch, mean=EURO_MEAN, std=EURO_STD):
    mean = torch.tensor(mean).view(1, 3, 1, 1)
    std = torch.tensor(std).view(1, 3, 1, 1)
    return (batch.cpu() * std + mean).clamp(0, 1)

sample_index = train_indices[0]
augmented = torch.stack([train_source[sample_index][0] for _ in range(10)])
grid = make_grid(denormalize(augmented), nrow=5)
plt.figure(figsize=(12, 5)); plt.imshow(grid.permute(1, 2, 0)); plt.axis("off")
plt.title("Десять обращений к одному train-примеру"); plt.show()


## 4. DataLoader: от примеров к batch

`DataLoader` не хранит данные заново: он выбирает индексы, вызывает Dataset, объединяет примеры в batch и может
загружать их параллельно.

- `batch_size`: компромисс между памятью, шумом градиента и скоростью;
- `shuffle=True`: меняет порядок train-примеров каждую эпоху;
- `num_workers`: процессы подготовки данных; в Colab обычно достаточно 2;
- `pin_memory=True`: ускоряет CPU→CUDA копирование;
- validation не перемешиваем: это упрощает воспроизводимость и анализ ошибок.


In [ ]:
batch_size = 128
loader_kwargs = dict(batch_size=batch_size, num_workers=2,
                     pin_memory=torch.cuda.is_available(), persistent_workers=True)
train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
xb, yb = next(iter(train_loader))
print("images:", tuple(xb.shape), xb.dtype, "| labels:", tuple(yb.shape), yb.dtype)
print("label range:", yb.min().item(), "…", yb.max().item(), "| batches/epoch:", len(train_loader))

grid = make_grid(denormalize(xb[:20]), nrow=10, padding=2)
plt.figure(figsize=(16, 5)); plt.imshow(grid.permute(1, 2, 0)); plt.axis("off")
plt.title("Один batch из DataLoader"); plt.show()
print([class_names[i] for i in yb[:20].tolist()])


## 5. Что делает свёртка

Один фильтр размера `K×K` скользит по изображению и использует одинаковые веса во всех позициях.
Для одной пространственной оси:

$$
out = \left\lfloor\frac{in + 2p - d(k-1) - 1}{s} + 1\right\rfloor
$$

Здесь `p` — padding, `s` — stride, `d` — dilation. `Conv2d(3, 16, 3, padding=1)` сохраняет H×W,
но заменяет 3 входных канала на 16 карт признаков.


In [ ]:
def conv_output_size(size, kernel=3, stride=1, padding=0, dilation=1):
    return (size + 2*padding - dilation*(kernel-1) - 1) // stride + 1

demo = torch.randn(4, 3, 64, 64)
conv = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
result = conv(demo)

print("Формула:", conv_output_size(32, kernel=3, padding=1))
print("PyTorch:", tuple(demo.shape), "→", tuple(result.shape))
print("Параметры:", sum(p.numel() for p in conv.parameters()), "= 16×3×3×3 + 16 bias")


In [ ]:
# Один вручную заданный edge-filter: положительный отклик на вертикальный переход.
patch = torch.tensor([[[[0., 0., 1., 1.],
                        [0., 0., 1., 1.],
                        [0., 0., 1., 1.],
                        [0., 0., 1., 1.]]]])
edge = nn.Conv2d(1, 1, 3, bias=False)
with torch.no_grad():
    edge.weight[:] = torch.tensor([[[[-1., 0., 1.],
                                      [-1., 0., 1.],
                                      [-1., 0., 1.]]]])
print("input:\n", patch[0, 0])
print("response:\n", edge(patch)[0, 0])


## 6. Архитектура


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )

    def forward(self, x):
        return self.block(x)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32, dropout=0.05),   # 64×64 → 32×32
            ConvBlock(32, 64, dropout=0.10),  # 32×32 → 16×16
            ConvBlock(64, 128, dropout=0.15), # 16×16 → 8×8
        )
        self.pool = nn.AdaptiveAvgPool2d(1)  # 8×8 → 1×1
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)  # raw logits


model = SmallCNN().to(device)
print(model)
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Forward hooks показывают изменение формы, не меняя forward().
hooks = []
def show_shape(name):
    def hook(_module, inputs, output):
        print(f"{name:>12}: {tuple(inputs[0].shape)} → {tuple(output.shape)}")
    return hook

for i, block in enumerate(model.features):
    hooks.append(block.register_forward_hook(show_shape(f"block {i+1}")))
hooks.append(model.pool.register_forward_hook(show_shape("global pool")))

with torch.inference_mode():
    logits = model(xb[:4].to(device))
print("logits:", tuple(logits.shape))
for hook in hooks:
    hook.remove()


## 7. Обучение

Важные переключатели:

- `model.train()` включает Dropout и обновление статистик BatchNorm;
- `model.eval()` отключает Dropout и фиксирует BatchNorm;
- `torch.inference_mode()` экономит память на validation;
- `CrossEntropyLoss` получает сырые logits и целочисленные labels.


In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = total_correct = total_examples = 0

    context = torch.enable_grad() if is_train else torch.inference_mode()
    with context:
        for inputs, targets in loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = loss_fn(logits, targets)
            if is_train:
                loss.backward()
                optimizer.step()

            total_examples += targets.size(0)
            total_loss += loss.item() * targets.size(0)
            total_correct += (logits.argmax(1) == targets).sum().item()

    return total_loss / total_examples, total_correct / total_examples


loss_fn = nn.CrossEntropyLoss(label_smoothing=0.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
epochs = 3 if FAST_MODE else 12
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_state, best_val_acc = None, -1.0

for epoch in range(epochs):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn)
    scheduler.step()
    for key, value in zip(history, [train_loss, train_acc, val_loss, val_acc]):
        history[key].append(value)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    print(f"epoch {epoch+1:02d}/{epochs} | train {train_loss:.3f}/{train_acc:.3f} "
          f"| val {val_loss:.3f}/{val_acc:.3f}")

model.load_state_dict(best_state)
print(f"Лучшая val accuracy: {best_val_acc:.3f}")


In [ ]:
epochs_axis = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_axis, history["train_loss"], "o-", label="train")
axes[0].plot(epochs_axis, history["val_loss"], "o-", label="val")
axes[0].set(title="Loss", xlabel="epoch"); axes[0].grid(); axes[0].legend()
axes[1].plot(epochs_axis, history["train_acc"], "o-", label="train")
axes[1].plot(epochs_axis, history["val_acc"], "o-", label="val")
axes[1].set(title="Accuracy", xlabel="epoch", ylim=(0, 1)); axes[1].grid(); axes[1].legend()
plt.show()


## 8. Метрики: что измеряет каждая

**Loss** оптимизируется градиентным спуском; её абсолютное значение зависит от loss function.

**Accuracy** — доля
верных top-1 предсказаний.

Для каждого класса: `precision = TP/(TP+FP)` отвечает «насколько чисты предсказания
класса», `recall = TP/(TP+FN)` — «сколько объектов класса нашли», `F1` — их гармоническое среднее.

**Macro** усредняет классы с одинаковым весом и замечает слабые классы.

**Weighted** взвешивает по support и ближе
к общей картине на несбалансированных данных.

**Top-k accuracy** полезна, когда несколько классов визуально близки.
Метрики считаем на всей эпохе, а не усредняем проценты отдельных batch разного размера.


In [ ]:
def collect_predictions(model, loader):
    model.eval(); image_parts, target_parts, prob_parts = [], [], []
    with torch.inference_mode():
        for inputs, targets in loader:
            probabilities = model(inputs.to(device)).softmax(1).cpu()
            image_parts.append(inputs); target_parts.append(targets); prob_parts.append(probabilities)
    return torch.cat(image_parts), torch.cat(target_parts), torch.cat(prob_parts)

images, targets, probabilities = collect_predictions(model, val_loader)
preds = probabilities.argmax(1); confidence = probabilities.max(1).values
num_classes = len(class_names)
confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)
for true, pred in zip(targets, preds): confusion[true, pred] += 1

tp = confusion.diag().float()
support = confusion.sum(1).float()
predicted = confusion.sum(0).float()
recall = tp / support.clamp_min(1)
precision = tp / predicted.clamp_min(1)
f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-12)
accuracy = tp.sum() / confusion.sum()
macro_f1 = f1.mean()
weighted_f1 = (f1 * support / support.sum()).sum()
top2 = probabilities.topk(2, dim=1).indices.eq(targets[:, None]).any(1).float().mean()

print(f"accuracy={accuracy:.3f} | macro-F1={macro_f1:.3f} | weighted-F1={weighted_f1:.3f} | top-2={top2:.3f}")
print(f"{'class':>22} {'precision':>9} {'recall':>8} {'F1':>7} {'support':>8}")
for name, p, r, score, n in zip(class_names, precision, recall, f1, support):
    print(f"{name:>22} {p:9.3f} {r:8.3f} {score:7.3f} {int(n):8d}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(confusion, cmap="Blues")
axes[0].set(xticks=range(num_classes), yticks=range(num_classes),
            xticklabels=class_names, yticklabels=class_names, xlabel="predicted", ylabel="true",
            title="Confusion matrix: строки=true, столбцы=prediction")
axes[0].tick_params(axis="x", rotation=60)
x = np.arange(num_classes); width = 0.25
axes[1].bar(x-width, precision.numpy(), width, label="precision")
axes[1].bar(x, recall.numpy(), width, label="recall")
axes[1].bar(x+width, f1.numpy(), width, label="F1")
axes[1].set(xticks=x, xticklabels=class_names, ylim=(0, 1), title="Метрики по классам")
axes[1].tick_params(axis="x", rotation=60); axes[1].legend(); axes[1].grid(axis="y")
plt.tight_layout(); plt.show()


In [ ]:
wrong = torch.where(preds != targets)[0]
wrong = wrong[torch.argsort(confidence[wrong], descending=True)][:16]
if len(wrong):
    grid = make_grid(denormalize(images[wrong]), nrow=8)
    plt.figure(figsize=(15, 5)); plt.imshow(grid.permute(1, 2, 0)); plt.axis("off"); plt.show()
    for idx in wrong.tolist():
        print(f"true={class_names[targets[idx]]:>22} | pred={class_names[preds[idx]]:>22} | conf={confidence[idx]:.2f}")


## 9. training toolbox

- **Bag of Tricks:** warmup, cosine schedule, корректная инициализация, weight decay, улучшенный preprocessing.
- **Label smoothing:** заменяет жёсткую one-hot цель сглаженной; в PyTorch: `CrossEntropyLoss(label_smoothing=0.1)`.
- **MixUp:** смешивает две картинки и их targets линейно; сглаживает границу решений.
- **CutMix:** вставляет прямоугольник другой картинки и смешивает targets пропорционально площади.
- **Random Erasing / Cutout:** скрывает участок и заставляет использовать несколько признаков.
- **RandAugment / TrivialAugment:** систематический выбор силы и состава аугментаций.
- **EMA/SWA:** усредняет веса во времени; часто улучшает стабильность и generalization.
- **Mixed precision:** ускоряет GPU-обучение и снижает расход памяти, но сама по себе не является регуляризацией.

Материалы: [Bag of Tricks](https://arxiv.org/abs/1812.01187),
[MixUp](https://arxiv.org/abs/1710.09412), [CutMix](https://arxiv.org/abs/1905.04899),
[label smoothing / Inception](https://arxiv.org/abs/1512.00567).


In [ ]:
# # Визуальная интуиция MixUp
# lam = 0.65
# permutation = torch.randperm(len(xb))
# mixed = lam * xb + (1 - lam) * xb[permutation]
# grid = make_grid(denormalize(mixed[:10]), nrow=5)
# plt.figure(figsize=(12, 5)); plt.imshow(grid.permute(1, 2, 0)); plt.axis("off")
# plt.title(f"MixUp: {lam:.2f}·image A + {1-lam:.2f}·image B"); plt.show()
# print("Loss: lam·CE(logits, y_a) + (1-lam)·CE(logits, y_b)")


## Практика

1. Посчитайте выходной размер для `in=32, kernel=5, stride=2, padding=2` и проверьте через `nn.Conv2d`.
2. Уберите второй `Conv2d` из каждого блока. Как изменятся число параметров, скорость и accuracy?
3. Замените `RandomVerticalFlip` на неподходящую аугментацию (например, сильный `RandomRotation`) и объясните результат.
4. Найдите пару классов с максимальной взаимной путаницей по confusion matrix.


## Мини-квиз

1. Зачем в свёртке weight sharing?  
2. Что меняет число `out_channels`?  
3. Почему `padding=1` у `kernel_size=3, stride=1` сохраняет размер?  
4. Почему validation нельзя запускать в `train()`?  
5. Почему accuracy недостаточно для диагностики?  
6. Что означает растущий train accuracy при падающем val accuracy?


## Подробнее

- [EuroSAT paper](https://arxiv.org/abs/1709.00029)
- [TorchVision: EuroSAT](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.EuroSAT.html)
- [PyTorch: Datasets and DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)
- [PyTorch: Transforms](https://docs.pytorch.org/tutorials/beginner/basics/transforms_tutorial.html)
- [PyTorch: Optimization loop](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)